This notebook covers the full data pipeline for the project:

1. **Data Ingestion**: Fetch raw race data from the OpenF1 API
2. **Feature Engineering**: Derive pit cost, max stint lengths, and the tire degradation model
3. **Model Fitting**: Build training datasets and fit the two logistic regression policy models used by Levin Tree Search

Run this notebook once before running `main.py`. All artifacts are saved to `data/`.

**Race**: 2024 Australian Grand Prix: `session_key=9488`, `meeting_key=1231`

## Imports & Config

In [ ]:
import json
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# ── Paths ──────────────────────────────────────────────────────────────────────
RAW_DIR        = "data/raw"
PARAM_DIR      = "data/parameter"
MODEL_DIR      = "data/model"
PATHS_DIR      = "data/paths"

for d in [RAW_DIR, PARAM_DIR, MODEL_DIR, PATHS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Race config ────────────────────────────────────────────────────────────────
SESSION_KEY  = 9488   # 2024 Australian GP Race session
MEETING_KEY  = 1231
TOTAL_LAPS   = 58
SAINZ_DRIVER = 55     # Carlos Sainz — race winner, used as model performance evaluation

## 1. Data Ingestion

Fetch laps, pit stops, and stints from the [OpenF1 API](https://openf1.org).
Results are stored locally in `data/raw/` so subsequent runs skip re-fetching.

In [ ]:
BASE_URL = "https://api.openf1.org/v1"

def ingest_data(endpoint: str, params: dict, cache_path: str) -> list[dict]:
    """
    Fetch a paginated OpenF1 endpoint and cache the result as JSON.
    If the cache file already exists, load from disk instead.
    """
    if os.path.exists(cache_path):
        print(f"Loading {cache_path}")
        with open(cache_path) as f:
            return json.load(f)

    url = f"{BASE_URL}/{endpoint}"
    print(f"GET {url}  params={params}")
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    with open(cache_path, "w") as f:
        json.dump(data, f, indent=2)
    print(f"Saved {len(data)} records → {cache_path}")
    return data

In [ ]:
common = {"session_key": SESSION_KEY, "meeting_key": MEETING_KEY}

raw_laps      = ingest_data("laps", common, f"{RAW_DIR}/laps.json")
raw_pit_stops = ingest_data("pit", common, f"{RAW_DIR}/pit-stops.json")
raw_stints    = ingest_data("stints", common, f"{RAW_DIR}/stints.json")

df_laps   = pd.DataFrame(raw_laps)
df_pits   = pd.DataFrame(raw_pit_stops)
df_stints = pd.DataFrame(raw_stints)

print(f"Laps:      {len(df_laps)} rows")
print(f"Pit stops: {len(df_pits)} rows")
print(f"Stints:    {len(df_stints)} rows")

### 1.1 Weather & Intervals

Fetch track/air temperature and interval (gap-ahead) data used by the
temperature-adjusted degradation model and the traffic penalty feature.

In [ ]:
raw_weather = ingest_data("weather", common, f"{RAW_DIR}/weather.json")
df_weather = pd.DataFrame(raw_weather)
df_weather["date"] = pd.to_datetime(df_weather["date"], utc=True, format="mixed")


In [ ]:
raw_intervals = ingest_data("intervals", common, f"{RAW_DIR}/intervals.json")
df_intervals = pd.DataFrame(raw_intervals)
df_intervals["date"] = pd.to_datetime(df_intervals["date"], utc=True, format="mixed")
df_intervals = df_intervals[["date", "driver_number", "interval"]].copy()


## 2. Feature Engineering

### 2.1 Pit Lane Time Loss

The pit lane time loss is the median `lane_duration` across all pit stops in the race.
This is used as a fixed cost added to any lap where a pit stop occurs.

In [ ]:
median_pit_loss = df_pits["lane_duration"].median()
print(f"Median pit lane time loss: {median_pit_loss:.3f}s")

### 2.2 Maximum Stint Lengths per Compound

Upper bound on how long a driver can run each compound. Used by the search to prune
unrealistic "continue" actions.

In [ ]:
df_stints["stint_length"] = df_stints["lap_end"] - df_stints["lap_start"] + 1
max_stints = df_stints.groupby("compound")["stint_length"].max()
print("Max stint lengths:")
print(max_stints.to_string())

max_stints.reset_index().rename(columns={"stint_length": "max_stint_length"}).to_csv(
    f"{PARAM_DIR}/max_stint_lengths.csv", index=False
)
print(f"\nSaved → {PARAM_DIR}/max_stint_lengths.csv")

### 2.3 Tire Degradation Model

For each `(compound, tire_age)` pair, estimate the expected lap time using the median
lap time across all drivers at that age. The pipeline:

1. Filter out pit-out laps, nulls, and possible safety car / VSC laps (95th-percentile cutoff)
2. Join laps with stints to compute `tire_age` per lap
3. Median lap time per `(compound, tire_age)` — drop groups with < 5 samples unless `tire_age ≤ 5`
4. 3-lap centered rolling average per compound (smoothing)
5. Interpolate missing ages within each compound range
6. Normalize: `relative_deg = smoothed_time - min(smoothed_time)` per compound

In [ ]:
def build_degradation_model(df_laps: pd.DataFrame, df_stints: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a tidy DataFrame with columns:
    compound, tire_age, expected_lap_time, sample_size,
    smoothed_time, base_time, relative_deg
    """
    # Step 1 — Filter
    valid = df_laps.dropna(subset=["lap_duration"]).copy()
    valid = valid[valid["is_pit_out_lap"] == False]
    upper = valid["lap_duration"].quantile(0.95)
    valid = valid[valid["lap_duration"] < upper]

    # Step 2 — Join with stints to get tire_age
    rows = []
    for _, lap in valid.iterrows():
        driver, lap_num = lap["driver_number"], lap["lap_number"]
        stint = df_stints[
            (df_stints["driver_number"] == driver) &
            (df_stints["lap_start"] <= lap_num) &
            (df_stints["lap_end"] >= lap_num)
        ]
        if stint.empty:
            continue
        stint = stint.iloc[0]
        tire_age = stint["tyre_age_at_start"] + (lap_num - stint["lap_start"])
        if tire_age <= 0:
            continue
        rows.append({"compound": stint["compound"], "tire_age": int(tire_age),
                     "lap_duration": lap["lap_duration"]})

    df_deg = pd.DataFrame(rows)

    # Step 3 — Aggregate
    agg = (
        df_deg
        .groupby(["compound", "tire_age"], as_index=False)
        .agg(expected_lap_time=("lap_duration", "median"),
             sample_size=("lap_duration", "count"))
    )
    agg = agg[(agg["sample_size"] >= 5) | (agg["tire_age"] <= 5)]

    # Step 4 — Smooth
    agg = agg.sort_values(["compound", "tire_age"]).copy()
    agg["smoothed_time"] = (
        agg.groupby("compound")["expected_lap_time"]
        .transform(lambda x: x.rolling(3, min_periods=1, center=True).mean())
    )

    # Step 5 — Interpolate missing ages
    agg = agg.set_index(["compound", "tire_age"])
    agg = agg.groupby(level=0).apply(
        lambda g: g.droplevel(0).reindex(
            range(1, int(g.index.get_level_values(1).max()) + 1)
        )
    ).interpolate().reset_index()

    # Step 6 — Normalize
    agg["base_time"] = agg.groupby("compound")["smoothed_time"].transform("min")
    agg["relative_deg"] = agg["smoothed_time"] - agg["base_time"]

    return agg

In [ ]:
deg_model = build_degradation_model(df_laps, df_stints)
print(deg_model.groupby("compound")[["tire_age","smoothed_time","relative_deg"]].describe())

deg_model.to_csv(f"{PARAM_DIR}/tire_degradation_model.csv", index=False)
print(f"\nSaved → {PARAM_DIR}/tire_degradation_model.csv")

In [ ]:
# Visualize degradation curves
fig, ax = plt.subplots(figsize=(9, 5))
colors = {"HARD": "lightgray", "MEDIUM": "yellow", "SOFT": "red"}
for comp, grp in deg_model.groupby("compound"):
    ax.plot(grp["tire_age"], grp["smoothed_time"],
            label=comp, color=colors.get(comp, "black"), linewidth=2)
ax.set_xlabel("Tire age (laps)")
ax.set_ylabel("Expected lap time (s)")
ax.set_title("Tire Degradation Model — 2024 Australian GP")
ax.legend()
plt.tight_layout()
plt.show()

### 2.4 Per-Lap Temperatures

Join weather to laps by nearest timestamp and derive the per-lap air/track
temperature used as a feature in the pit-decision and compound-choice policy models.

In [ ]:
df_laps["date"] = pd.to_datetime(df_laps["date_start"], utc=True, format="mixed")
df_weather_sorted = df_weather.sort_values("date")
df_laps_sorted = df_laps.dropna(subset=["date"]).sort_values("date")

df_laps_with_temp = pd.merge_asof(
    df_laps_sorted,
    df_weather_sorted[["date", "air_temperature", "track_temperature"]],
    on="date",
    direction="nearest",
)
per_lap_temps = (
    df_laps_with_temp
    .groupby("lap_number")[["air_temperature", "track_temperature"]]
    .mean()
    .reset_index()
    .rename(columns={"lap_number": "lap"})
)
per_lap_temps.to_csv(f"{PARAM_DIR}/per_lap_temperatures.csv", index=False)
print(f"Saved -> {PARAM_DIR}/per_lap_temperatures.csv")
per_lap_temps.head()


### 2.5 Dirty Air Penalty

Under 2022 ground-effect regulations, running within a 1-second interval behind
another car costs ~0.49 s/lap in time loss ([Source](https://www.formuladream.app/blog/f1-traffic-measured)). 

Rather than a one-time gap-at-pit-exit penalty, we model the **cumulative dirty-air cost**
over the laps following a pit stop: for each lap in the horizon window where the
driver's interval to the car ahead is inside the threshold, a per-lap coefficient
is applied and summed, stopping early once the driver escapes into clear air.

In [ ]:
df_laps_valid = df_laps.dropna(subset=["lap_duration", "date_start"]).copy()
df_laps_valid["date_start"] = pd.to_datetime(df_laps_valid["date_start"], utc=True, format="mixed")
df_laps_valid["date_end"]   = df_laps_valid["date_start"] + pd.to_timedelta(
    df_laps_valid["lap_duration"], unit="s"
)

interval_records = []
for _, lap_row in df_laps_valid.iterrows():
    mask = (
        (df_intervals["driver_number"] == lap_row["driver_number"]) &
        (df_intervals["date"] >= lap_row["date_start"]) &
        (df_intervals["date"] <= lap_row["date_end"])
    )
    lap_intervals = df_intervals.loc[mask, "interval"].dropna()
    if lap_intervals.empty:
        continue
    interval_records.append({
        "driver_number": lap_row["driver_number"],
        "lap_number":    lap_row["lap_number"],
        "median_interval": lap_intervals.median(),
    })

df_lap_intervals = pd.DataFrame(interval_records)
print(f"Built per-lap interval medians: {len(df_lap_intervals)} rows")

In [ ]:
DIRTY_AIR_THRESHOLD = 1.0
DIRTY_AIR_COEFF     = 0.49
DIRTY_AIR_HORIZON   = 15

# Identify each pit stop: driver + pit lap (lap before the pit-out lap)
pit_events = df_laps[df_laps["is_pit_out_lap"] == True][
    ["driver_number", "lap_number"]
].copy()
pit_events["pit_lap"] = pit_events["lap_number"] - 1

dirty_air_rows = []
for _, pit in pit_events.iterrows():
    driver  = pit["driver_number"]
    pit_lap = int(pit["pit_lap"])
    total_penalty = 0.0

    for L in range(pit_lap + 1, pit_lap + DIRTY_AIR_HORIZON + 1):
        row = df_lap_intervals[
            (df_lap_intervals["driver_number"] == driver) &
            (df_lap_intervals["lap_number"] == L)
        ]
        if row.empty:
            break  # driver has finished the race or data missing 
        interval_val = row.iloc[0]["median_interval"]
        if pd.isna(interval_val):
            continue  # driver is leading 
        lap_penalty = DIRTY_AIR_COEFF * max(0.0, 1.0 - interval_val / DIRTY_AIR_THRESHOLD)
        total_penalty += lap_penalty
        if interval_val >= DIRTY_AIR_THRESHOLD:
            break  # driver has escaped dirty air 

    dirty_air_rows.append({
        "driver_number": driver,
        "pit_lap":       pit_lap,
        "dirty_air_cost": round(total_penalty, 3),
    })

df_dirty_air = pd.DataFrame(dirty_air_rows)
print(df_dirty_air[df_dirty_air["dirty_air_cost"] > 0].to_string())

In [ ]:
df_traffic = (
    df_dirty_air
    .groupby("pit_lap")["dirty_air_cost"]
    .median()
    .reset_index()
    .rename(columns={"pit_lap": "lap", "dirty_air_cost": "traffic_penalty"})
)

df_traffic.to_csv(f"{PARAM_DIR}/traffic_penalties.csv", index=False)
print(f"Saved {len(df_traffic)} rows → {PARAM_DIR}/traffic_penalties.csv")
print(df_traffic[df_traffic["traffic_penalty"] > 0].to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(df_traffic["lap"], df_traffic["traffic_penalty"], color="steelblue")
ax.set_xlabel("Pit lap")
ax.set_ylabel("Dirty air cost (s)")
ax.set_title(f"Cumulative dirty air penalty per pit lap\n"
             f"(COEFF={DIRTY_AIR_COEFF}s/lap, threshold={DIRTY_AIR_THRESHOLD}s, "
             f"horizon={DIRTY_AIR_HORIZON} laps)")
plt.tight_layout()
plt.show()

## 3. Model Fitting

Two logistic regression models provide the policy for Levin Tree Search:
- `model_pit`:  should the driver pit on this lap or continue
- `model_comp`: which compound does the driver switch to

### 3.1 Build Training Datasets

In [ ]:
tire_lookup = deg_model.set_index(["compound", "tire_age"])["expected_lap_time"].to_dict()


def build_pit_dataset(df_stints: pd.DataFrame, total_laps: int = 58) -> pd.DataFrame:
    """One row per lap-in-stint. Binary target: did the driver pit at the end of this lap?"""
    rows = []
    df = df_stints.sort_values(["meeting_key", "session_key", "driver_number", "stint_number"])
    for _, stint in df.iterrows():
        for lap in range(stint.lap_start, stint.lap_end + 1):
            tire_age = stint.tyre_age_at_start + (lap - stint.lap_start)
            action = 1 if (lap == stint.lap_end and lap < total_laps) else 0
            elt = tire_lookup.get((stint.compound, tire_age),
                                  deg_model[deg_model["compound"] == stint.compound]["expected_lap_time"].mean())
            rows.append({
                "lap": lap,
                "laps_remaining": total_laps - lap,
                "tire_age": tire_age,
                "compound": stint.compound,
                "is_soft":   1 if stint.compound == "SOFT"   else 0,
                "is_medium": 1 if stint.compound == "MEDIUM" else 0,
                "is_hard":   1 if stint.compound == "HARD"   else 0,
                "expected_lap_time": elt,
                "pit": action,
            })
    return pd.DataFrame(rows)


def build_compound_dataset(df_stints: pd.DataFrame, total_laps: int = 58) -> pd.DataFrame:
    """One row per inter-stint transition. Target: next compound chosen."""
    rows = []
    df = df_stints.sort_values(["meeting_key", "session_key", "driver_number", "stint_number"])
    for _, group in df.groupby(["meeting_key", "session_key", "driver_number"]):
        group = group.sort_values("stint_number").reset_index(drop=True)
        for i in range(len(group) - 1):
            cur  = group.iloc[i]
            nxt  = group.iloc[i + 1]
            lap  = cur.lap_end
            if lap >= total_laps:
                continue
            tire_age = cur.tyre_age_at_start + (lap - cur.lap_start)
            rows.append({
                "lap": lap,
                "laps_remaining": total_laps - lap,
                "tire_age": tire_age,
                "compound_before": cur.compound,
                "is_soft_before":   1 if cur.compound == "SOFT"   else 0,
                "is_medium_before": 1 if cur.compound == "MEDIUM" else 0,
                "is_hard_before":   1 if cur.compound == "HARD"   else 0,
                "next_compound": nxt.compound,
            })
    return pd.DataFrame(rows)

In [ ]:
df_pit      = build_pit_dataset(df_stints, TOTAL_LAPS)
df_compound = build_compound_dataset(df_stints, TOTAL_LAPS)

print(f"Pit dataset:      {len(df_pit)} rows  |  pit=1: {df_pit['pit'].sum()}")
print(f"Compound dataset: {len(df_compound)} rows")
print("\nNext compound distribution:")
print(df_compound["next_compound"].value_counts())

# Save
df_pit.to_csv(f"{MODEL_DIR}/pit_dataset.csv", index=False)
df_compound.to_csv(f"{MODEL_DIR}/pit_compound_dataset.csv", index=False)
print(f"\nSaved datasets → {MODEL_DIR}/")

### 3.2 Fit Models

In [ ]:
# Pit decision model (binary)
X_pit = df_pit.drop(columns=["compound", "pit", "is_soft"])
y_pit = df_pit["pit"]

model_pit = LogisticRegression(solver="newton-cholesky", class_weight="balanced", max_iter=1000)
model_pit.fit(X_pit, y_pit)

print("=== Pit Decision Model ===")
print(classification_report(y_pit, model_pit.predict(X_pit), target_names=["continue", "pit"]))

In [ ]:
# Compound choice model (multiclass)
X_comp = df_compound.drop(columns=["compound_before", "next_compound", "is_soft_before"])
y_comp = df_compound["next_compound"]

model_comp = LogisticRegression(solver="newton-cholesky", class_weight="balanced", max_iter=1000)
model_comp.fit(X_comp, y_comp)

print("=== Compound Choice Model ===")
print(classification_report(y_comp, model_comp.predict(X_comp)))

### 3.3 Feature Importances (Coefficients)

In [ ]:
coef_pit = pd.Series(model_pit.coef_[0], index=X_pit.columns).sort_values()
coef_pit.plot(kind="barh", figsize=(7, 4), title="Pit Decision Model — Coefficients")
plt.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

## 4. Data Output Summary

| Artifact | Path |
|---|---|
| Raw laps | `data/raw/laps.json` |
| Raw pit stops | `data/raw/pit-stops.json` |
| Raw stints | `data/raw/stints.json` |
| Raw weather | `data/raw/weather.json` |
| Raw intervals | `data/raw/intervals.json` |
| Max stint lengths | `data/parameter/max_stint_lengths.csv` |
| Tire degradation model | `data/parameter/tire_degradation_model.csv` |
| Per-lap temperatures | `data/parameter/per_lap_temperatures.csv` |
| Traffic penalties | `data/parameter/traffic_penalties.csv` |
| Pit decision dataset | `data/model/pit_dataset.csv` |
| Compound choice dataset | `data/model/pit_compound_dataset.csv` |